# valentine — introductory notebook

M3 requires a minimal notebook per selected tool. This one covers **valentine**, the
library under study: load two tables, run a matcher, read the result object, score it
against documented ground truth, and register a trivial matcher of your own.

The project uses Valentine as the baseline suite; the extension in
`src/swsn/extension/` is a `BaseMatcher` subclass that plugs into the same harness.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

## Two tables

Loaded through the project's table loader so the path, encoding and sampling match every
other notebook and the benchmark. Both tables come from Olist and are documented as
joinable on `order_id`.

In [2]:
from swsn.tables import load_table

orders = load_table("olist", "olist_orders_dataset", data_dir="../data", nrows=5000)
payments = load_table("olist", "olist_order_payments_dataset", data_dir="../data", nrows=5000)
print("orders  ", orders.shape)
print("payments", payments.shape)
print("orders columns  :", list(orders.columns))
print("payments columns:", list(payments.columns))

orders   (5000, 8)
payments (5000, 5)
orders columns  : ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
payments columns: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']


## Run a matcher

`valentine_match` takes a list of tables, a matcher instance and names for the tables.
It returns a `MatcherResults`, which is a read-only mapping from `ColumnPair` to a score.
Coma is hybrid here: schema (name) similarity and instance (value) similarity.

In [3]:
from valentine import valentine_match
from valentine.algorithms import Coma

matches = valentine_match(
    [orders, payments],
    Coma(use_schema=True, use_instances=True),
    df_names=["orders", "payments"],
)
print(f"{len(matches)} candidate pairs")
for pair, score in list(matches.items())[:5]:
    print(f"{pair.source_column:35} -> {pair.target_column:25} {score:.3f}")

5 candidate pairs
order_id                            -> order_id                  0.502
order_delivered_customer_date       -> payment_value             0.197
order_purchase_timestamp            -> payment_value             0.196
order_approved_at                   -> payment_value             0.191
order_delivered_carrier_date        -> payment_value             0.191


## Reading `MatcherResults`

The mapping is sorted from highest to lowest score, so the first item is the matcher's
best proposal. `get_details(pair)` returns the per-sub-matcher breakdown when the matcher
provides one — for Coma, the name and instance components.

In [4]:
best = next(iter(matches))
print("best pair :", best)
print("score     :", round(matches[best], 3))
print("details   :", matches.get_details(best))

best pair : ColumnPair(source_table='orders', source_column='order_id', target_table='payments', target_column='order_id')
score     : 0.502
details   : {'NameCM': 1.0, 'InstancesCM': 0.005}


## Score against documented ground truth

The publisher documents which columns correspond, so the ground truth does not have to be
hand-annotated. `ground_truth_set` returns `(source_column, target_column)` pairs for one
table pair; Valentine's metrics accept full `ColumnPair` instances, which keeps the table
names in the comparison.

In [5]:
from swsn.schema_doc import load_schema, documented_correspondences, ground_truth_set

schema = load_schema("../ground_truth/schema/olist.yaml")
correspondences = documented_correspondences(schema)
gt_pairs = ground_truth_set(
    correspondences, "olist_orders_dataset", "olist_order_payments_dataset"
)
print("documented pairs for orders x payments:", sorted(gt_pairs))

documented pairs for orders x payments: [('order_id', 'order_id')]


In [6]:
from valentine.algorithms import ColumnPair
from valentine.metrics import Precision, Recall, F1Score

ground_truth = [ColumnPair("orders", a, "payments", b) for a, b in gt_pairs]
for metric in (Precision(), Recall(), F1Score()):
    print(metric.apply(matches, ground_truth))

{'Precision': 0.5}
{'Recall': 1.0}
{'F1Score': 0.6666666666666666}


The numbers look harsh because this table pair has **one** documented correspondence, so
precision and recall are coarse: one wrong proposal among the candidates halves
precision. The benchmark uses twelve pairs and reports by stratum for exactly this reason
(`evaluation/protocol.yaml`). The default `one_to_one=True` applies Hungarian selection;
pass `one_to_one=False` to score the raw proposal set instead.

## Register your own matcher

Every matcher implements one method, `get_matches(source_input, target_input)`, and returns
a dict of `ColumnPair` to score. The batch method has a default that calls `get_matches`
over every pair of tables, so a two-table matcher needs nothing else. This one proposes a
pair only when the two column names are identical — the simplest possible baseline.

In [7]:
from valentine.algorithms import BaseMatcher, ColumnPair

class ExactNameMatcher(BaseMatcher):
    """Propose a pair only when the two column names are identical."""

    def get_matches(self, source_input, target_input):
        pairs = {}
        for source_column in source_input.get_columns():
            for target_column in target_input.get_columns():
                if source_column.name == target_column.name:
                    pairs[
                        ColumnPair(
                            source_input.name, source_column.name,
                            target_input.name, target_column.name,
                        )
                    ] = 1.0
        return pairs

exact = valentine_match(
    [orders, payments], ExactNameMatcher(), df_names=["orders", "payments"]
)
print(f"{len(exact)} exact-name matches")
for pair, score in exact.items():
    print(pair.source_column, "->", pair.target_column, score)

1 exact-name matches
order_id -> order_id 1.0


## What this shows

- Valentine's contract is small: implement `get_matches`, return `ColumnPair` scores.
  Everything else — metrics, one-to-one selection, details — is provided by the library.
- The extension differs from `ExactNameMatcher` only in how it computes the score:
  COMA's name functions plus Jaccard over canonicalised value sets, with the two
  components exposed in `match_details`.
- A single table pair is not enough to judge a matcher; that is why the project evaluates
  on twelve pairs, five corruption levels and per stratum.

## Reproducing

```bash
make data     # the Olist CSVs
make eda      # execute this notebook
```